[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/niccoloridi/International-Law-and-AI/blob/main/IntlLaw_AI_Lab03_Hardware.ipynb)

In [ ]:
#@title Lab 03 — The Hardware Behind the Hype
print("""
 █████             █████    █████                                                     █████████   █████
░░███             ░░███    ░░███                                         ███         ███░░░░░███ ░░███ 
 ░███  ████████   ███████   ░███         ██████   █████ ███ █████       ░███        ░███    ░███  ░███ 
 ░███ ░░███░░███ ░░░███░    ░███        ░░░░░███ ░░███ ░███░░███     ███████████    ░███████████  ░███ 
 ░███  ░███ ░███   ░███     ░███         ███████  ░███ ░███ ░███    ░░░░░███░░░     ░███░░░░░███  ░███ 
 ░███  ░███ ░███   ░███ ███ ░███      █ ███░░███  ░░███████████         ░███        ░███    ░███  ░███ 
 █████ ████ █████  ░░█████  ███████████░░████████  ░░████░████          ░░░         █████   █████ █████
░░░░░ ░░░░ ░░░░░    ░░░░░  ░░░░░░░░░░░  ░░░░░░░░    ░░░░ ░░░░                      ░░░░░   ░░░░░ ░░░░░ 
                                                                                                       
                                                                                                       
                                                                                                       

   Lab 03 // The Hardware Behind the Hype
   Melbourne Law Masters 2026
""")

*Brought to you by **Dr Niccolò Ridi**, Purveyor of Fine Scripts* – [KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)

# The Hardware Behind the Hype

## Overview

AI is often treated as pure software – elegant algorithms, clever code, brilliant ideas. But **AI has a body**. It lives in silicon, in data centres, in undersea cables, in the geopolitical chokepoints of Taiwan's semiconductor foundries.

In this lab, we make the *physical infrastructure of AI visible*. You will:

1. **Understand matrix multiplication** – the mathematical primitive that powers every neural network
2. **See why GPUs crushed CPUs** – and what that means for compute concentration
3. **Count the FLOPs** – measure AI's appetite for computation, and trace that appetite to real geopolitical conflicts
4. **Map AI's geography** – where chips are designed, manufactured, and the resource costs of training frontier models
5. **Connect hardware to international law** – export controls, environmental obligations, sovereignty, and the politics of compute

By the end, you'll understand why controlling semiconductors is effectively controlling AI – and why that matters for international law.

## Table of Contents

> **Part One: Matrix Multiplication – The Engine of AI**
>> [The fundamental operation](#scrollTo=part_one_intro)
>> [From 2×2 to 10,000×10,000](#scrollTo=matrix_scaling)
>> [Visualising neural network weights](#scrollTo=matrix_viz)

> **Part Two: CPU vs GPU – Why GPUs Won**
>> [The architecture story](#scrollTo=part_two_intro)
>> [Your Colab hardware](#scrollTo=hardware_check)
>> [The GPU race: CPU vs GPU benchmarks](#scrollTo=gpu_race)
>> [Interactive matrix multiplication race](#scrollTo=interactive_race)

> **Part Three: Counting Compute – FLOPs and Scaling Laws**
>> [What are FLOPs?](#scrollTo=flops_intro)
>> [Computing the cost of frontier models](#scrollTo=model_costs)
>> [Chinchilla scaling laws](#scrollTo=chinchilla)
>> [Training GPT-4: an economic reality](#scrollTo=gpt4_cost)

> **Part Four: The Geography of AI**
>> [Where chips come from](#scrollTo=geography_intro)
>> [The supply chain](#scrollTo=supply_chain)
>> [Environmental costs](#scrollTo=environmental)
>> [Policy reflection](#scrollTo=policy_reflection)

> **Part Five: Hardware as International Law**
>> [Export controls and sovereignty](#scrollTo=intl_law_hw)
>> [Environmental obligations](#scrollTo=env_obligations)
>> [The geopolitics of compute](#scrollTo=geopolitics)

> **Review Quiz**
>> [Test your understanding](#scrollTo=quiz)

# Part One // Matrix Multiplication – The Engine of AI

<a id='part_one_intro'></a>

## The Fundamental Operation

Every neural network – whether GPT-4, Gemini, or DALL-E – is ultimately just **matrix multiplication**.

When you feed a prompt into ChatGPT:
1. Your words are converted to vectors (embeddings)
2. These vectors are multiplied by weight matrices (the "knowledge" learned during training)
3. The output is multiplied by more weight matrices
4. After thousands of these operations, you get your answer

The forward pass through GPT-3 (175 billion parameters) involves approximately **350 trillion matrix multiplications** for a single prompt. No wonder hardware matters.

Let's start simple.

In [ ]:
#@title ## Simple 2×2 Matrix Multiplication: By Hand vs NumPy

import numpy as np
import time

#@markdown We'll multiply two 2×2 matrices:
#@markdown $$A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}, \quad B = \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix}$$

# Define matrices
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("Matrix A:")
print(A)
print("\nMatrix B:")
print(B)

#@markdown **Manual calculation of C = A × B:**
#@markdown - C[0,0] = (1×5) + (2×7) = 5 + 14 = 19
#@markdown - C[0,1] = (1×6) + (2×8) = 6 + 16 = 22
#@markdown - C[1,0] = (3×5) + (4×7) = 15 + 28 = 43
#@markdown - C[1,1] = (3×6) + (4×8) = 18 + 32 = 50

C_manual = np.array([[19, 22], [43, 50]])
print("\nManual result:")
print(C_manual)

#@markdown **NumPy's result:**
C_numpy = np.matmul(A, B)
print("\nNumPy result:")
print(C_numpy)

print("\n✓ They match! This simple operation is the atom of all neural networks.")

<a id='matrix_scaling'></a>

## Scaling Up: From 2×2 to 10,000×10,000

Now let's see what happens as we increase the matrix size. This is the story of AI compute in a nutshell.

In [ ]:
#@title ## Matrix Multiplication: Size and Speed

import matplotlib.pyplot as plt

#@markdown Let's multiply increasingly large square matrices and measure the time taken.

sizes = [100, 500, 1000, 2000, 5000, 10000]
times = []

print("Multiplying square matrices A[n×n] × B[n×n]...\n")
print(f"{'Size':<10} {'Time (ms)':<15} {'Relative Speed':<15}")
print("-" * 40)

for size in sizes:
    A = np.random.randn(size, size)
    B = np.random.randn(size, size)
    
    start = time.time()
    C = np.matmul(A, B)
    elapsed = (time.time() - start) * 1000  # Convert to milliseconds
    
    times.append(elapsed)
    relative = elapsed / times[0]
    
    print(f"{size:<10} {elapsed:<15.3f} {relative:<15.1f}x")

#@markdown Notice: at 10,000×10,000, a single operation takes ~10 seconds on CPU. That's trillions of multiplications.

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale
ax1.plot(sizes, times, 'o-', linewidth=2, markersize=8, color='#e74c3c')
ax1.set_xlabel('Matrix Size (n)', fontsize=12)
ax1.set_ylabel('Time (ms)', fontsize=12)
ax1.set_title('Matrix Multiplication Time (Linear Scale)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Log scale
ax2.loglog(sizes, times, 'o-', linewidth=2, markersize=8, color='#3498db')
ax2.set_xlabel('Matrix Size (n)', fontsize=12)
ax2.set_ylabel('Time (ms)', fontsize=12)
ax2.set_title('Matrix Multiplication Time (Log Scale)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("\n📊 Key insight: Time grows as O(n³). A 100× increase in size → ~1,000,000× increase in time.")

<a id='matrix_viz'></a>

## Visualising Neural Network Weights

A neural network weight matrix is what the model has "learned". Let's visualise one.

In [ ]:
#@title ## Heatmap of a Neural Network Weight Matrix

#@markdown A weight matrix after training. Each cell is a learned parameter, typically initialized randomly
#@markdown and then adjusted during training. The pattern you see is the result of backpropagation
#@markdown across millions of examples.

# Create a realistic weight matrix
np.random.seed(42)
W = np.random.normal(0, 0.1, size=(256, 256))  # Typical hidden layer size

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(W, cmap='coolwarm', aspect='auto')
ax.set_xlabel('Output Neuron', fontsize=11)
ax.set_ylabel('Input Neuron', fontsize=11)
ax.set_title('Weight Matrix: 256 inputs → 256 outputs\n(A typical layer in a modern transformer)', 
             fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Weight Value', fontsize=11)

plt.tight_layout()
plt.show()

print(f"This matrix contains {256 * 256:,} parameters.")
print(f"GPT-3 has 175 *billion* parameters across thousands of such matrices.")
print(f"\nIn memory, this single matrix occupies: {(256 * 256 * 4) / (1024**2):.2f} MB (assuming 32-bit floats)")
print(f"GPT-3 occupies: ~{(175e9 * 4) / (1024**3):.0f} GB in memory.")

# Part Two // CPU vs GPU – Why GPUs Won

<a id='part_two_intro'></a>

## The Architecture Story

**CPUs** are designed for *flexibility*:
- Few cores (maybe 8-16 in a laptop)
- Each core is "smart" – can do many different things
- Built for sequential logic, branching, complex operations
- Good for: Word processors, web browsers, anything general-purpose

**GPUs** are designed for *parallelism*:
- Thousands of cores (Tesla T4 has 2,560 cores; H100 has 14,080)
- Each core is "simple" – does the same thing as all the others
- Built for doing the same operation on millions of data points simultaneously
- Good for: Graphics (hence the name), and – it turns out – matrix multiplication

Matrix multiplication is **embarrassingly parallel**: every output element is computed independently. This is *exactly* what GPUs excel at.

In 2012, Geoffrey Hinton's team used GPUs to win the ImageNet competition by a landslide. The AI boom was built on that discovery: GPUs are orders of magnitude faster at neural networks than CPUs.

<a id='hardware_check'></a>

## What Hardware Has Colab Given Us?

In [ ]:
#@title ## Check Your Colab Hardware

import torch
import subprocess

print("=" * 70)
print("CHECKING AVAILABLE HARDWARE")
print("=" * 70)

#@markdown Check if GPU is available:
gpu_available = torch.cuda.is_available()
print(f"\nGPU Available: {gpu_available}")

if gpu_available:
    device_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    print(f"GPU Type: {device_name}")
    print(f"Number of GPUs: {device_count}")
    
    # Get more details with nvidia-smi
    print("\n" + "=" * 70)
    print("NVIDIA-SMI OUTPUT")
    print("=" * 70)
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        print(result.stdout)
    except:
        print("(nvidia-smi not available)")
else:
    print("\n⚠️  No GPU detected. This lab will run on CPU, but slower.")
    print("(In a paid Colab environment, you can enable GPU support.)")

print("\nCPU Details:")
print(f"PyTorch CPU threads: {torch.get_num_threads()}")

<a id='gpu_race'></a>

## The GPU Race: CPU vs GPU Benchmarks

Let's run the same matrix multiplication on both CPU and GPU (if available) and see the difference.

In [ ]:
#@title ## CPU vs GPU Matrix Multiplication Benchmark

import torch
import time
import matplotlib.pyplot as plt

#@markdown We'll multiply square matrices of increasing size on both CPU and GPU.
#@markdown If no GPU is available, we'll gracefully fall back to CPU-only timing.

sizes = [500, 1000, 2000, 5000, 10000]
cpu_times = []
gpu_times = []

gpu_available = torch.cuda.is_available()

print("Benchmarking matrix multiplication on CPU vs GPU...\n")
if gpu_available:
    gpu_device = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_device}\n")
else:
    print("GPU: Not available (will show CPU only)\n")

print(f"{'Size':<10} {'CPU (ms)':<15} {'GPU (ms)':<15} {'Speedup':<10}")
print("-" * 50)

for size in sizes:
    # CPU computation
    A_cpu = torch.randn(size, size)
    B_cpu = torch.randn(size, size)
    
    # Warmup (often the first run is slower)
    _ = torch.matmul(A_cpu, B_cpu)
    
    start = time.time()
    C_cpu = torch.matmul(A_cpu, B_cpu)
    cpu_time = (time.time() - start) * 1000
    cpu_times.append(cpu_time)
    
    # GPU computation
    if gpu_available:
        A_gpu = A_cpu.cuda()
        B_gpu = B_cpu.cuda()
        
        # Warmup
        _ = torch.matmul(A_gpu, B_gpu)
        torch.cuda.synchronize()  # Ensure GPU is done before timing
        
        start = time.time()
        C_gpu = torch.matmul(A_gpu, B_gpu)
        torch.cuda.synchronize()
        gpu_time = (time.time() - start) * 1000
        gpu_times.append(gpu_time)
        
        speedup = cpu_time / gpu_time
        print(f"{size:<10} {cpu_time:<15.3f} {gpu_time:<15.3f} {speedup:<10.1f}x")
    else:
        print(f"{size:<10} {cpu_time:<15.3f} {'N/A':<15} {'N/A':<10}")

if gpu_available:
    # Plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = range(len(sizes))
    width = 0.35
    
    bars1 = ax.bar([i - width/2 for i in x], cpu_times, width, label='CPU', color='#3498db')
    bars2 = ax.bar([i + width/2 for i in x], gpu_times, width, label=f'GPU ({gpu_device})', color='#e74c3c')
    
    ax.set_xlabel('Matrix Size (n)', fontsize=12)
    ax.set_ylabel('Time (ms)', fontsize=12)
    ax.set_title('CPU vs GPU: Matrix Multiplication Speed', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(sizes)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add speedup labels
    for i, (cpu_t, gpu_t) in enumerate(zip(cpu_times, gpu_times)):
        speedup = cpu_t / gpu_t
        ax.text(i, max(cpu_t, gpu_t) * 1.05, f'{speedup:.0f}x', ha='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    print("\n✓ GPU dominates for large matrices. But notice: for small matrices, the overhead of GPU communication")
    print("  might make CPU competitive (not visible here due to our benchmark size).")
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sizes, cpu_times, 'o-', linewidth=2, markersize=8, color='#3498db', label='CPU')
    ax.set_xlabel('Matrix Size (n)', fontsize=12)
    ax.set_ylabel('Time (ms)', fontsize=12)
    ax.set_title('Matrix Multiplication: CPU Only', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

<a id='interactive_race'></a>

## Interactive: Design Your Own Matrix Multiplication Race

In [ ]:
#@title ## Interactive Matrix Multiplication Race

import torch
import time

matrix_size = 2000 #@param {type:"slider", min:100, max:10000, step:100}

#@markdown Choose a matrix size above. We'll race CPU vs GPU.

print(f"Racing: {matrix_size} × {matrix_size} matrix multiplication\n")
print("=" * 60)

A_cpu = torch.randn(matrix_size, matrix_size)
B_cpu = torch.randn(matrix_size, matrix_size)

# CPU
_ = torch.matmul(A_cpu, B_cpu)  # Warmup
start = time.time()
C_cpu = torch.matmul(A_cpu, B_cpu)
cpu_time = (time.time() - start) * 1000

print(f"\n🖥️  CPU Time: {cpu_time:.2f} ms")

if torch.cuda.is_available():
    A_gpu = A_cpu.cuda()
    B_gpu = B_cpu.cuda()
    
    _ = torch.matmul(A_gpu, B_gpu)  # Warmup
    torch.cuda.synchronize()
    start = time.time()
    C_gpu = torch.matmul(A_gpu, B_gpu)
    torch.cuda.synchronize()
    gpu_time = (time.time() - start) * 1000
    
    print(f"🚀 GPU Time: {gpu_time:.2f} ms")
    print(f"\n🏆 GPU is {cpu_time/gpu_time:.1f}x faster!")
    print("\n" + "=" * 60)
    print(f"\nContext: A single forward pass through GPT-3 involves ~350 trillion")
    print(f"matrix multiplications. At GPU speed ({gpu_time:.2f} ms per {matrix_size}×{matrix_size}),")
    print(f"that's why frontier models require massive GPU farms.")
else:
    print("\n(GPU not available)")
    print(f"\nContext: A single forward pass through GPT-3 involves ~350 trillion")
    print(f"matrix multiplications. At CPU speed ({cpu_time:.2f} ms per {matrix_size}×{matrix_size}),")
    print(f"you can see why AI researchers switched to GPUs.")

# Part Three // Counting Compute – FLOPs and Scaling Laws

<a id='flops_intro'></a>

## What Are FLOPs?

**FLOP** = Floating-Point Operation (one multiplication or addition)

We measure AI compute in FLOPs per second (FLOP/s), because it's hardware-independent.

**For matrix multiplication** of two n×n matrices:
- Each of the n² output cells requires n multiplications + (n-1) additions
- Total: roughly **2n³ FLOPs**

**Example**: 10,000 × 10,000 matrix multiplication ≈ 2 × 10¹² FLOPs

### Hardware Scales
- **CPU** (laptop): ~10-100 GFLOP/s (10¹⁰ - 10¹¹)
- **GPU** (consumer, RTX 4090): ~90 TFLOP/s (9 × 10¹³)
- **NVIDIA H100**: ~1.6 PFLOP/s (1.6 × 10¹⁵) for FP32 precision
- **A100**: ~650 TFLOP/s
- **T4 (free Colab)**: ~65 TFLOP/s

A H100 can do in **one second** what a laptop CPU would take **16 million seconds** to do (≈6 months).

<a id='model_costs'></a>

## Computing the Cost of Frontier Models

In [ ]:
#@title ## The Compute Requirements of Modern AI Models

import pandas as pd
import numpy as np

#@markdown Below are estimates of the compute required for training frontier models.
#@markdown These are **training** FLOPs; inference (what happens when you use ChatGPT) is cheaper.

models = {
    'Model': ['GPT-2', 'GPT-3', 'GPT-3.5', 'Gemini Ultra', 'GPT-4', 'Claude 3 Opus'],
    'Parameters': [1.5e9, 175e9, 175e9, '~1.5T*', '~1.8T*', '~100B*'],
    'Training FLOPs (est.)': [3.2e21, 1.3e24, 2.4e24, '~2e25*', '~2.15e25*', '~1e24*'],
    'GPU-Years (H100)': [0.06, 500, 900, '~6,300*', '~6,700*', '~3,100*'],
    'Est. Cost ($M)': [10, 100, 150, '~1,000*', '~100-200*', '~50*'],
}

df = pd.DataFrame(models)
print("\nFrontier AI Model Training Costs")
print("=" * 100)
print(df.to_string(index=False))
print("\n* Estimates; exact figures not officially disclosed.")
print("\nAssumptions:")
print("- H100 GPU: 1.6 PFLOP/s (FP32), costs ~$40k, lasts ~5 years")
print("- Effective cost per FLOP: ~$10^-10 (hardware amortized + electricity + data centre)")
print("- Training efficiency: ~20-50% of theoretical peak (real-world overhead)")

print("\n" + "=" * 100)
print("\n🔑 Key insight: A single frontier model costs $100M-$1B to train.")
print("This concentrates AI power in the hands of well-funded labs (Google, OpenAI, Anthropic, Meta, etc.).")
print("Smaller organizations cannot afford to train frontier models from scratch.")

<a id='chinchilla'></a>

## Chinchilla Scaling Laws

In 2022, DeepMind researchers (Hoffmann et al.) published the "Chinchilla" scaling laws, which describe the optimal relationship between:
- **Compute** (C): the total FLOPs spent training
- **Model size** (P): the number of parameters
- **Dataset size** (D): the number of tokens in the training data

**The finding**: To optimally use C FLOPs, you should:
- Allocate roughly **equal compute** to model parameters and data
- Model size ≈ Data size (in tokens)

This contradicted earlier assumptions that models were "undertrained" (i.e., we could use more data). It showed that the scaling limits are not model size, but *total compute available*.

**Implication**: If you can afford 10²⁵ FLOPs of compute, you should train a model with ~10¹¹ parameters on ~10¹¹ tokens. The constraint is the GPU farm, not the model architecture.

In [ ]:
#@title ## Chinchilla Scaling Law Visualization

import matplotlib.pyplot as plt
import numpy as np

#@markdown The Chinchilla scaling law predicts that model performance (loss) decreases
#@markdown as you increase compute, following a power law.

# Synthetic data following Chinchilla scaling (L ∝ C^-0.07 approximately)
compute_values = np.logspace(21, 26, 50)  # FLOPs from 10^21 to 10^26
loss_values = 3.0 * (compute_values / 1e24) ** (-0.07)  # Power law

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-log plot
ax1.loglog(compute_values, loss_values, 'o-', linewidth=2, markersize=6, color='#2ecc71')
ax1.set_xlabel('Training Compute (FLOPs)', fontsize=11)
ax1.set_ylabel('Validation Loss (lower is better)', fontsize=11)
ax1.set_title('Chinchilla Scaling Law: Loss vs Compute', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, which='both')

# Add annotations for known models
models_annotated = [
    (1.3e24, 2.95, 'GPT-3'),
    (2.15e25, 2.75, 'GPT-4 (est.)'),
]
for compute, loss, label in models_annotated:
    ax1.plot(compute, loss, 'r*', markersize=15)
    ax1.annotate(label, (compute, loss), xytext=(10, 10), textcoords='offset points',
                fontsize=10, bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

# Relationship between model size and compute (optimal allocation)
model_sizes = np.logspace(9, 13, 50)  # Parameters from 10^9 to 10^13
optimal_compute = 20 * model_sizes  # Chinchilla: C ≈ 20 * P

ax2.loglog(model_sizes, optimal_compute, 'o-', linewidth=2, markersize=6, color='#9b59b6')
ax2.set_xlabel('Model Size (Parameters)', fontsize=11)
ax2.set_ylabel('Optimal Compute (FLOPs)', fontsize=11)
ax2.set_title('Chinchilla: Optimal Compute Allocation', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, which='both')

# Annotations
known_models = [
    (1.5e9, 'GPT-2'),
    (175e9, 'GPT-3'),
    (1.8e12, 'GPT-4 (est.)'),
]
for params, label in known_models:
    compute = 20 * params
    ax2.plot(params, compute, 'r*', markersize=15)
    ax2.annotate(label, (params, compute), xytext=(10, 10), textcoords='offset points',
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n📊 Chinchilla Insights:")
print("\n1. Scaling is predictable: Loss decreases as C^-0.07")
print("2. Optimal model size: P_optimal ≈ C / 20")
print("3. Implications:")
print("   - To double model performance, increase compute by ~2^(1/0.07) ≈ 1,000×")
print("   - Doubling model size requires 20× more compute to train optimally")
print("   - The main constraint is compute budget, not model architecture")

<a id='gpt4_cost'></a>

## Training GPT-4: An Economic Reality

In [ ]:
#@title ## The Economics of Training GPT-4

#@markdown GPT-4 training details (OpenAI, March 2023):

# Estimates from various sources
flops_gpt4 = 2.15e25  # FLOPs
h100_throughput = 1.6e15  # FLOP/s

# Time to compute
seconds_needed = flops_gpt4 / h100_throughput
hours_needed = seconds_needed / 3600
days_needed = hours_needed / 24
years_needed = days_needed / 365

print("Training GPT-4: The Math")
print("=" * 70)
print(f"\nEstimated training FLOPs: {flops_gpt4:.2e}")
print(f"NVIDIA H100 throughput: {h100_throughput:.2e} FLOP/s")
print(f"\nTime on a single H100: {years_needed:.1f} years")
print(f"\nWith 1,000 H100 GPUs: {years_needed/1000:.2f} years ≈ {days_needed/1000:.0f} days")

# Cost
h100_price = 40000  # USD per GPU
num_gpus = 1000
total_hardware_cost = h100_price * num_gpus

# Electricity (rough estimate)
power_per_h100 = 700  # Watts
total_power = power_per_h100 * num_gpus / 1000  # kW
training_hours = hours_needed / 1000  # With 1000 GPUs
electricity_cost = (total_power * training_hours) * 0.10  # $0.10 per kWh (data centre rate)

# Data centre overhead
data_centre_cost = total_hardware_cost * 2  # Rough: facility, cooling, network, etc.

print(f"\n" + "=" * 70)
print("Cost Breakdown (with 1,000 H100 GPUs):")
print("=" * 70)
print(f"\nGPU hardware (1,000 × $40k): ${total_hardware_cost:,.0f}")
print(f"Electricity (~{training_hours:.0f} hours): ${electricity_cost:,.0f}")
print(f"Data centre infrastructure (2×): ${data_centre_cost:,.0f}")
print(f"\nTotal estimate: ${total_hardware_cost + electricity_cost + data_centre_cost:,.0f}")
print(f"\nIn reality, OpenAI likely spent $50M-$200M on GPT-4 training.")
print(f"(Plus additional costs for fine-tuning, RLHF training, etc.)")

print(f"\n" + "=" * 70)
print("🔑 What This Means for International Law")
print("=" * 70)
print(f"""
1. CAPITAL CONCENTRATION
   - Only organizations with $100M-$500M can train frontier models.
   - This is a handful of US tech companies + a few backed by national governments.
   - This shapes whose values, biases, and priorities are embedded in AI.

2. COMPUTE HEGEMONY
   - Compute = power. Those who control GPU supply chains control AI development.
   - Export controls on GPUs (US → China) are effectively AI arms control.
   - Taiwan's TSMC: the chokepoint of global AI infrastructure.

3. GEOGRAPHIC CONCENTRATION
   - Training happens where electricity is cheap: Virginia, Oregon, Middle East.
   - This creates environmental and sovereignty questions.
   - Data centres require massive water usage: hyperscalers in water-stressed regions.

4. POWER ASYMMETRIES
   - Europe cannot match US/China compute budgets → dependent on US/Chinese models.
   - Developing countries have no path to frontier AI without accessing US companies.
   - This is a new form of technological dependency.
""")

# Part Four // The Geography of AI

<a id='geography_intro'></a>

## Where AI Hardware Comes From

AI is not made in Silicon Valley labs. It's made in:

1. **Taiwan (TSMC)**: Manufactures 95%+ of advanced semiconductor chips (5nm and below)
2. **South Korea (Samsung, SK Hynix)**: Memory (VRAM, HBM) – critical for GPU performance
3. **California (NVIDIA, AMD, Intel)**: Design GPUs, CPUs
4. **Netherlands, Japan**: Specialized equipment (ASML lithography tools)
5. **China**: Growing (SMIC), but blocked from access to advanced nodes by US export controls

**The irony**: The most advanced AI chips in the world are manufactured on an island that the PRC claims sovereignty over. This is not accidental geopolitics – it's the foundation of AI geopolitics.

Also: **The minerals**. Semiconductors require rare earth elements, cobalt, nickel – mostly from politically unstable regions or controlled by specific states (especially China).

<a id='supply_chain'></a>

## The Supply Chain

In [ ]:
#@title ## AI Chip Supply Chain Map

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

#@markdown The global supply chain for AI accelerators (GPUs).

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Define supply chain stages
stages = [
    {'name': 'Raw Materials', 'x': 1, 'y': 8, 'color': '#e74c3c', 'items': ['Rare Earths (China)\nCobalt (Congo)\nNickel (Indonesia)']},
    {'name': 'Equipment Makers', 'x': 3, 'y': 8, 'color': '#f39c12', 'items': ['ASML (Netherlands)\nLithography tools']},
    {'name': 'Chip Designers', 'x': 5, 'y': 8, 'color': '#3498db', 'items': ['NVIDIA (USA)\nAMD (USA)\nQualcomm (USA)']},
    {'name': 'Chip Manufacturers', 'x': 7, 'y': 8, 'color': '#9b59b6', 'items': ['TSMC (Taiwan)\nSamsung (SK)\nIntel (USA)']},
    {'name': 'Integrators', 'x': 9, 'y': 8, 'color': '#1abc9c', 'items': ['Google, Meta\nOpenAI, Anthropic\nMicrosoft']},
    {'name': 'Training Locations', 'x': 5, 'y': 4, 'color': '#16a085', 'items': ['USA: N. Virginia\nOregon, California\n\nME: UAE, Saudi Arabia\nEurope: Limited\nChina: Isolated']},
]

# Draw boxes
box_width = 1.5
box_height = 1.2

for stage in stages:
    x, y = stage['x'], stage['y']
    # Box
    fancy_box = FancyBboxPatch((x - box_width/2, y - box_height/2), 
                               box_width, box_height,
                               boxstyle="round,pad=0.1", 
                               edgecolor='black', facecolor=stage['color'], alpha=0.7)
    ax.add_patch(fancy_box)
    
    # Title
    ax.text(x, y + 0.3, stage['name'], ha='center', va='center', 
           fontsize=10, fontweight='bold')
    # Items
    for i, item in enumerate(stage['items']):
        ax.text(x, y - 0.15 - i*0.25, item, ha='center', va='center', 
               fontsize=7, style='italic')

# Draw arrows between stages
for i in range(len(stages) - 2):
    x1 = stages[i]['x'] + 0.8
    x2 = stages[i+1]['x'] - 0.8
    y = 8
    arrow = FancyArrowPatch((x1, y), (x2, y),
                           arrowstyle='->', lw=2, color='black', alpha=0.6)
    ax.add_patch(arrow)

# Arrow to training locations
arrow = FancyArrowPatch((5, 7.3), (5, 5),
                       arrowstyle='<->', lw=2, color='#c0392b', linestyle='dashed')
ax.add_patch(arrow)

ax.text(4.2, 6.1, 'Shipped\nWorldwide', fontsize=9, style='italic', color='#c0392b')

# Add geopolitical annotations
ax.text(1, 6.5, '⚠️ CHINA\nControls supply\nof rare earths', 
       fontsize=8, bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
ax.text(7, 6.5, '⚠️ TAIWAN\n95% of advanced\nchips made here', 
       fontsize=8, bbox=dict(boxstyle='round', facecolor='orange', alpha=0.8))
ax.text(3.5, 0.5, '⚠️ US EXPORT CONTROLS\nAdvanced chips banned to China (EUV, 5nm+)', 
       fontsize=9, bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.8))

ax.set_title('Global AI Chip Supply Chain: A Geopolitical Chokepoint', 
            fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CHOKEPOINT ANALYSIS")
print("="*70)
print("""
CRITICAL DEPENDENCIES:

1. TAIWAN (TSMC)
   - Makes 95% of world's advanced chips
   - Single island, geopolitically disputed
   - China's "return to homeland" would cripple global AI
   - US strategic interest: defending TSMC = defending AI supremacy

2. NETHERLANDS (ASML)
   - Only manufacturer of EUV lithography machines (needed for 5nm+)
   - One company, one country
   - Dutch government can restrict sales (tried with China)

3. RARE EARTH ELEMENTS
   - 95% of global supply controlled by China
   - Used in GPUs, storage, power electronics
   - China has leveraged this before (rare earth embargo on Japan, 2010)

4. MEMORY (VRAM, HBM)
   - SK Hynix, Samsung (South Korea); Micron (US)
   - HBM (High Bandwidth Memory) is bottleneck for AI accelerators
   - Much of it still manufactured in South Korea

GEOPOLITICAL CONSEQUENCES:

- US has weaponized chip exports (EUV chips to China banned since 2022)
- EU trying to build domestic capacity (CHIPS Act, €43B) — but still dependent
- Taiwan's security = global AI security
- China building indigenous alternative (SMIC), but 5-10 years behind
- Chip shortage of 2020-2021 showed fragility of supply chain
""")

<a id='environmental'></a>

## Environmental Costs of AI Infrastructure

In [ ]:
#@title ## The Environmental Footprint of AI

import matplotlib.pyplot as plt
import numpy as np

#@markdown AI's environmental impact is often hidden. Let's make it visible.

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

# 1. Energy consumption of major data centres
data_centres = ['Google\nData Centers', 'Meta\n(Facebook)', 'Microsoft\nAzure', 'AWS\n(Amazon)', 'Apple']
energy_twh = [18, 7, 7, 12, 8]  # TWh per year (estimates)

colors_energy = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#95a5a6']
ax1.bar(data_centres, energy_twh, color=colors_energy, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Annual Energy Consumption (TWh)', fontsize=11, fontweight='bold')
ax1.set_title('Data Centre Energy Consumption\n(Hyperscalers)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(energy_twh):
    ax1.text(i, v + 0.3, f'{v}', ha='center', fontweight='bold')

# 2. Carbon emissions from training models
models = ['GPT-2', 'GPT-3', 'BERT\n(large)', 'Stable\nDiffusion', 'GPT-4\n(est.)']
co2_tonnes = [11, 550, 1400, 6000, 25000]  # Estimated CO2e in tonnes

ax2.barh(models, co2_tonnes, color=['#2ecc71', '#f39c12', '#e74c3c', '#c0392b', '#8b0000'], 
         alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_xlabel('CO2 Equivalent (tonnes)', fontsize=11, fontweight='bold')
ax2.set_title('Carbon Footprint of Training Models', fontsize=12, fontweight='bold')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3, axis='x', which='both')

# Add context labels
for i, v in enumerate(co2_tonnes):
    ax2.text(v * 1.2, i, f'{v:,}', va='center', fontweight='bold')

ax2.axvline(x=4600, color='green', linestyle='--', linewidth=2, label='Annual per capita (USA)')
ax2.axvline(x=1200, color='blue', linestyle='--', linewidth=2, label='Annual per capita (Global)')
ax2.legend(fontsize=9, loc='lower right')

# 3. Water consumption
data_sources = ['Data Centre\nCooling', 'Chip\nManufacturing', 'Rare Earth\nMining', 'Total AI\nInfrastructure']
water_gallons = [500000, 7000000, 250000, 7750000]  # Gallons per H100 trained (rough estimate)
water_liters = [x * 3.785 for x in water_gallons]

ax3.bar(data_sources, water_liters, color=['#3498db', '#2980b9', '#1c5ba0', '#0a3a52'], 
        alpha=0.7, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Water Consumption (Litres)', fontsize=11, fontweight='bold')
ax3.set_title('Water Usage: Training a Frontier Model\n(Approximate for H100 farm)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
for i, v in enumerate(water_liters):
    ax3.text(i, v + 300000, f'{v/1e6:.1f}M', ha='center', fontweight='bold')

ax3.axhline(y=9461538, color='red', linestyle='--', linewidth=2, label='Olympic pool volume')
ax3.legend(fontsize=9)

# 4. Energy source of data centres
energy_sources = ['Coal', 'Natural Gas', 'Nuclear', 'Renewables\n(Wind, Solar)']
enviro_good = [10, 15, 20, 55]  # % of Google's energy (most progressive)
enviro_bad = [40, 35, 10, 15]   # % of global average

x = np.arange(len(energy_sources))
width = 0.35

ax4.bar(x - width/2, enviro_good, width, label='Google (est.)', color='#2ecc71', alpha=0.7, edgecolor='black')
ax4.bar(x + width/2, enviro_bad, width, label='Global Average', color='#e74c3c', alpha=0.7, edgecolor='black')

ax4.set_ylabel('Percentage of Total Energy', fontsize=11, fontweight='bold')
ax4.set_title('Energy Mix: Data Centres', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(energy_sources)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')
ax4.set_ylim(0, 70)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ENVIRONMENTAL FACTS & CONCERNS")
print("="*70)
print("""
1. ENERGY
   - A single forward pass through GPT-3: ~0.4 kWh (1 query costs as much electricity
     as manufacturing a smartphone uses)
   - Training GPT-3: 1,287 MWh (equivalent to powering 130 US homes for a year)
   - Data centres consume 1-2% of global electricity
   - This will grow: AI demand doubling every 3-4 months

2. CARBON
   - Training GPT-4: estimated 25,000 tonnes CO2e (equivalent to ~5,200 cars for a year)
   - This assumes clean energy; global average is much worse
   - Inference (using ChatGPT): tiny per-query carbon (0.0004g CO2 per 100 queries)
   - But trillions of queries = significant total carbon

3. WATER
   - Data centre cooling uses enormous amounts of water
   - Google uses ~5 billion gallons/day globally
   - This puts pressure on water-stressed regions
   - Some AI data centres in Middle East (UAE, Saudi Arabia) despite water scarcity

4. PHYSICAL WASTE
   - GPU hardware lifespan: 3-5 years
   - Electronic waste: rare earth extraction is environmentally destructive
   - Mining for rare earths in Congo, China: massive environmental damage

5. GEOGRAPHIC INJUSTICE
   - Rare earth mining in Congo: environmental disaster, human rights abuses
   - Chip manufacturing in Taiwan: high water usage in an island
   - Data centres in Middle East: energy-intensive in carbon-intensive regions
   - Developed nations benefit from AI; developing nations bear environmental costs

INTERNATIONAL LAW ANGLE:
   - Paris Agreement commitments: should include AI infrastructure carbon
   - Common but differentiated responsibilities: who pays for AI's emissions?
   - Environmental justice: costs in Global South, benefits in Global North
   - Right to water: data centres vs. human access to clean water
   - Climate litigation: will governments be liable for AI's carbon?

REFERENCES:
   - BBC: "The Hidden Environmental Costs of AI" (2023)
   - ICJ Advisory Opinion on Climate Change (2023)
   - Strubell et al., "Energy and Policy Considerations for Deep Learning in NLP" (2019)
""")

<a id='policy_reflection'></a>

## Policy Reflection: Should International Law Regulate AI Hardware?

In [ ]:
#@title ## Reflection Question: Regulating AI Hardware

response = "Undecided" #@param ["Yes, comprehensive control", "Limited controls only", "No, market should decide", "Undecided"]

print("\n" + "="*70)
print("DEBATE: Should International Law Regulate AI Hardware?")
print("="*70)

print(f"\nYour position: {response}")

print("""

🏛️ ARGUMENTS FOR COMPREHENSIVE INTERNATIONAL REGULATION:

1. ARMS CONTROL PRECEDENT
   - Nuclear weapons are controlled by NPT (Non-Proliferation Treaty)
   - Chips for AI are effectively AI weapons
   - A global treaty on AI semiconductor trade could prevent an "arms race"

2. ENVIRONMENTAL PROTECTION
   - AI data centres' carbon footprint should be regulated like other industries
   - Water-intensive cooling could be restricted in arid regions
   - Right to sustainable environment (affirmed in UN Resolution 2022)

3. FAIRNESS / EQUITY
   - Concentrate power when chips are scarce; currently only 5 companies can train frontier models
   - International regulation could democratize access
   - Developing nations have a claim to fair distribution of compute

4. SUPPLY CHAIN RESILIENCE
   - Taiwan's dominance is a systemic risk
   - International agreement could fund redundant, distributed manufacturing
   - (CHIPS Act is unilateral attempt; multilateral version would be better)

5. LABOR & HUMAN RIGHTS
   - Chip manufacturing has labour issues (Malaysia, Philippines)
   - Mining for rare earths in Congo has human rights abuses
   - International standards could improve working conditions

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📈 ARGUMENTS AGAINST REGULATION (Market-Friendly View):

1. INNOVATION RISK
   - Regulation slows down chip development
   - US CHIPS Act subsidy is better than control
   - Letting companies compete drives efficiency

2. ENFORCEMENT DIFFICULTY
   - Semiconductors can be smuggled
   - Dual-use technology (civilians + military)
   - Impossible to verify compliance globally

3. SOVEREIGNTY CONCERNS
   - Companies have right to sell (profit motive)
   - Nations resist external control (Taiwan won't accept UN chip quotas)
   - Tech nationalism: each country wants independence

4. ECONOMIC COMPLEXITY
   - AI chips valuable; restricting supply increases prices
   - Developing nations would pay more (regressive)
   - Energy efficiency improvements matter more than output limits

5. MARKET SELF-CORRECTION
   - Competition will solve supply chain issues
   - Samsung, Intel developing alternatives to TSMC
   - Restrictions create artificial scarcity

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🤔 MIDDLE GROUND: "LIGHT TOUCH" REGULATION

1. ENVIRONMENTAL STANDARDS
   - Require data centre carbon reporting (like financial disclosure)
   - Tax on high-energy compute (carbon tax)
   - Standards for water recycling
   - No outright bans, but market incentives

2. SUPPLY CHAIN TRANSPARENCY
   - Mandated disclosure of where chips are made
   - Human rights due diligence for rare earth mining
   - Similar to conflict minerals regulations

3. EXPORT CONTROLS (Status Quo)
   - Keep US restrictions on advanced chips to China/Iran
   - But make it transparent and negotiable (not unilateral)
   - MTCR (Missile Technology Control Regime) as model

4. RESEARCH FUNDING
   - Support open-access compute via international consortium
   - UNESCO-style body that allocates compute time fairly
   - Similar to CERN model for physics

5. LABOR/HUMAN RIGHTS
   - Mandatory audits of chip manufacturing conditions
   - Supply chain responsibility (Apple model, extended to all)
   - International labour standards, enforced

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

❓ QUESTIONS FOR YOU:

1. If chips = weapons, does semiconductor regulation = arms control?
   → Does that change the answer?

2. Should compute be treated as a public good (like nuclear energy)
   or a private commodity (like oil)?

3. Taiwan's vulnerability: is it a reason to regulate globally?
   Or a reason to keep it out of international agreements?

4. Environmental justice: who should bear the cost of AI's carbon footprint?
   (Users? Developers? Nations where data centres are located?)

5. What would international law look like?
   - UN treaty? Regional agreements? Industry standards?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📚 FOR NEXT SESSION:

Connect this to Session 5 (Regulation & Governance):
- How do current AI regulations (EU AI Act) address compute concentration?
- Should there be "compute minimums" for AI development (like environmental impact assessments)?

Connect to Session 6 (Military AI):
- Control over chips = control over military AI capabilities
- Taiwan contingency: what if China takes Taiwan?
- Semiconductor independence as national security strategy
""")

# Part Five // Hardware as International Law

<a id='intl_law_hw'></a>

## Export Controls and Sovereignty

**The US Semiconductor Export Ban to China (2022-present)**

In October 2022, the US Commerce Department announced strict export controls on advanced semiconductors (5nm and below) to China. This was *not* a trade dispute – it was **AI policy**.

**Why it matters**:
- Advanced chips are necessary for training frontier AI models
- China's AI capabilities depend on accessing US/NVIDIA technology
- By restricting chips, the US effectively restricts China's AI development
- This is arms control for the 21st century, but without a treaty

**Legal framework**: Export controls technically fall under "international trade law" but are exercised unilaterally under **national security** exceptions (allowed under WTO rules).

**Questions for international law**:
1. Is this legitimate? (US says: yes, national security; China says: imperialism)
2. Should there be a global treaty on AI semiconductor exports?
3. Can export controls on chips violate WTO rules, or is "national security" a blank check?
4. What about allied nations? (Japan, Netherlands also restrict sales to China)

**The Taiwan Question**: Taiwan's Semiconductor Manufacturing Company (TSMC) makes 95% of advanced chips. If China invades Taiwan, the entire global AI supply chain collapses. This is the geopolitical story of our time.

<a id='env_obligations'></a>

## Environmental Obligations: From Paris to the ICJ

The **2023 ICJ Advisory Opinion on Climate Change** (requested by Vanuatu and the Alliance of Small Island States) confirmed that:

1. States have an obligation to protect the global climate (customary international law)
2. This obligation applies to *new* technologies, including AI
3. Failure to act on climate change can constitute a breach of human rights law

**How this applies to AI hardware**:

- **Data centre carbon emissions**: Should be counted toward Paris Agreement commitments
- **Water consumption**: Violates right to water (enshrined in UN Resolution 64/292)
- **Rare earth mining**: Environmental destruction in vulnerable regions → climate justice issues
- **Intergenerational justice**: Future generations inherit depleted resources

**No international law currently addresses AI infrastructure's environmental impact specifically.** This is a regulatory gap.

Possible instruments:
- Amendments to Paris Agreement to include "digital infrastructure carbon"
- New protocol on AI and climate (like cyber treaties are being negotiated)
- Right to repair: extend devices' lifespans to reduce e-waste

<a id='geopolitics'></a>

## The Geopolitics of Compute: Implications for Sovereignty and Power

## Geopolitics Summary: Who Controls AI?

THE GEOPOLITICS OF AI COMPUTE: Who Controls Tomorrow's Intelligence?

1️⃣  COMPUTE HEGEMONY: The "Big Three"

   🇺🇸 UNITED STATES
    NVIDIA (GPU design): 95% market share
    AMD, Intel (x86 CPUs, competing GPUs)
    OpenAI, Google, Meta, Anthropic (model training)
    TSMC manufacturing (de facto US ally)
    Strategic advantage: Controls the full stack

   🇨🇳 CHINA
    Huawei, SMIC (chip design + manufacturing, but restricted)
    Alibaba, Baidu, Tencent (models)
    BLOCKED: Cannot import advanced US chips (export ban)
    Strategy: Build indigenous ecosystem, but 5-10 years behind

   🇪🇺 EUROPEAN UNION
    ASML (lithography equipment, critical chokepoint)
    Some chip manufacturing (but not leading edge)
    AI models: Smaller players (Aleph Alpha, etc.)
    Strategy: CHIPS Act (€43B) to build capacity, but dependent on US for design

---

2️⃣  THE CHOKEPOINTS

   🏭 TSMC (Taiwan)
      - Makes 95% of world's advanced chips
      - Located on island claimed by China
      - Single point of failure for global AI
      - Geopolitical flashpoint: China's military buildup around Taiwan
      - US security commitment: defend Taiwan = defend AI supply chain
      - Scenario: If Taiwan falls to China:
         → Global AI development halts for 18-24 months
         → $5 trillion+ economic damage
         → Military balance shifts decisively to China

   🇳🇱 ASML (Netherlands)
      - Only manufacturer of EUV lithography equipment (needed for 5nm chips)
      - Dutch government can restrict exports
      - Already has: restricted sales to China under US pressure
      - China tried to develop alternatives (Huada Semiconductor): ~5 years behind

   🇨🇳 Rare Earth Elements
      - China controls 95% of global supply
      - Used in magnets, semiconductors, batteries
      - China leveraged this: 2010 embargo on Japan (rare earth shortage)
      - Possible future leverage: China restricts exports during US-China conflict

   🇰🇷 Memory (VRAM, HBM)
      - SK Hynix, Samsung (South Korea); Micron (US)
      - HBM (High Bandwidth Memory) is GPU bottleneck
      - Korea vulnerable to China pressure + North Korean military threat

---

3️⃣  IMPLICATIONS FOR INTERNATIONAL LAW

   📋 TECHNOLOGY TRANSFER & ESPIONAGE
      - China's semiconductor espionage: stealing chip designs from TSMC, Samsung
      - US sanctions: China's inability to legally access chips → sanctions evasion
      - International law gap: No clear rules for "cyber-espionage of IP"
      → Question: Is stealing AI chip designs an act of war? (Stuxnet-like?)

   ⚔️ ARMS CONTROL ANALOGY
      - Compare to nuclear weapons:
        - NPT controls access to fissile material
        - US controls access to advanced chips (de facto AI arms control)
        - But there's no treaty, no verification, no shared framework
      → Gap: Need international semiconductor control regime (like MTCR for missiles)

   🌍 SOVEREIGNTY & DEPENDENCY
      - Europe dependent on US for chip design (NVIDIA, design tools)
      - Europe dependent on Taiwan for manufacturing
      - Europe dependent on Netherlands for equipment
      → Result: Limited strategic autonomy in AI
      → EU CHIPS Act response: try to build domestic capacity
      → But decades away from competing with TSMC

   🏛️ TRADE VS. SECURITY
      - WTO rules allow trade exceptions for "national security"
      - But what counts as national security?
      → US: AI chips are national security (export ban justified)
      → China: Export ban is unfair trade practice
      → EU: Caught in middle, pressured to align with US
      → Developing nations: Can't afford to buy restricted chips anyway

   💦 WATER & ENVIRONMENTAL RIGHTS
      - Data centres use massive amounts of water
      - Located in water-stressed regions (Arizona, Middle East)
      → Question: Right to water vs. right to AI access?
      → Issue: No international law addresses AI infrastructure's water impact

---

4️⃣  SCENARIOS: Where Does This Go?

   🔴 SCENARIO A: "Great Power Competition"
      - US-China rivalry over AI chips intensifies
      - Taiwan becomes flashpoint; geopolitical crisis
      - Supply chains fragment into "US bloc" and "China bloc"
      - EU tries to stay neutral but forced to choose
      - Result: Slower AI development globally, competing standards
      → International law impact: New treaties on tech control, arms race dynamics

   🟡 SCENARIO B: "Negotiated Settlement"
      - US and China negotiate AI supply chain agreement
      - Limited export controls, but with transparency
      - Similar to Cold War arms control (SALT, START treaties)
      - Taiwan's status negotiated (unlikely, but possible)
      → International law impact: New AI treaty framework, like nuclear treaties

   🟢 SCENARIO C: "Distributed Independence"
      - EU, India, Japan, others build indigenous chip capacity
      - ASML alternatives developed (China, EU, Japan competing)
      - No single chokepoint, redundant supply chains
      - Takes 10+ years and $100B+ investment
      → International law impact: Regional frameworks instead of global

   ⚫ SCENARIO D: "Status Quo"
      - US maintains hegemony through unilateral export controls
      - No formal international law, just great power pressure
      - TSMC remains de facto "in US sphere"
      - Developing nations have limited access
      → International law impact: None; norms emerge through practice (customary law)

---

5️⃣  FOR INTERNATIONAL LAWYERS: Key Questions

   ❓ Should there be a "Semiconductors Non-Proliferation Treaty" (SNPT)?
      - Models: NPT (nuclear), MTCR (missiles), Wassenaar (dual-use exports)
      - Challenges: Verification, enforcement, rapid technological change

   ❓ What counts as "national security" in export controls?
      - Current: US broad interpretation (any AI chip to China is restricted)
      - Alternative: Narrower definition (only military AI, not civilian)
      - Question: Who decides?

   ❓ Do environmental impacts create legal obligations?
      - ICJ climate opinion: yes, states have climate duty
      - Implication: AI data centres' carbon = state responsibility
      - Question: Can NGOs sue governments for AI's environmental damage?

   ❓ Is Taiwan's status an AI law issue?
      - Realpolitik: TSMC independence = global AI resilience
      - Legal question: Does international law care about manufacturing location?
      - Precedent: Global supply chain fragmentation (2020-2021 taught lessons)

   ❓ What about developing nations?
      - They can't buy restricted chips; they can't build their own
      - Is this a violation of right to development (UN principle)?
      - Should international law guarantee "AI access for all nations"?

---

6️⃣  CONNECTIONS TO OTHER SESSIONS

   📍 Session 1 (Basics): AI = computation. Control computation = control AI.

   📍 Session 2 (Law Fundamentals): Export controls, national security exceptions,
      treaty-making authority.

   📍 Session 5 (Regulation): How to regulate AI? Can't separate from compute regulation.
      EU CHIPS Act, US CHIPS Act, China's indigenous development.

   📍 Session 6 (Military AI): Who has military AI? Whoever has the chips.
      Ukraine conflict shows why compute matters (drone swarms, targeting).

   📍 Session 7 (Data & Privacy): Data centres' location = jurisdiction. Where your data
      is processed = which country's law applies.

---

---

## End of Lab 03: The Hardware Behind the Hype

**What you've learned**:
- Matrix multiplication is the atom of AI
- GPUs enabled the AI boom through parallelism
- Frontier model training costs $50M-$500M (compute, not data)
- AI infrastructure has a geography: TSMC, ASML, US
- Hardware is now a vector of great power competition
- Environmental costs are rising and unregulated
- International law has no coherent response

**For your essays / discussions**:
- Should compute be regulated like nuclear energy or left to markets?
- What happens to international law if Taiwan falls to China?
- How should the environmental costs of AI be distributed globally (Paris Agreement)?
- Is semiconductor export control legitimate arms control, or imperialism?

---

**Dr. Niccolò Ridi**  
Melbourne Law Masters 2026  
[KCL Profile](https://www.kcl.ac.uk/people/niccolo-ridi)